# 🧪 Notebook 03 — Experiments (Essayage de tous les modèles)
**Projet :** Cervical Cancer Risk Prediction  
**Auteure :** Hadil Dhaya · 4th Year Data Science · Group 5

---
**Objectifs :**
- Tester **7 modèles** de classification
- Comparer via AUC-ROC, F1, Recall, Precision, Accuracy
- Cross-validation 5-fold stratifiée
- Identifier le meilleur modèle
- Tuning des hyperparamètres du meilleur
- Sauvegarder le meilleur modèle

> ⚠️ Ce notebook est le **cahier d'expériences** — toutes les tentatives y sont.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               AdaBoostClassifier)
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.model_selection import (
    cross_val_score, StratifiedKFold, RandomizedSearchCV
)

warnings.filterwarnings('ignore')
os.makedirs('../outputs', exist_ok=True)
print('✅ Imports OK')

## 1. Chargement des données prétraitées

In [ ]:
X_train = pd.read_csv('../outputs/X_train.csv')
X_test  = pd.read_csv('../outputs/X_test.csv')
y_train = pd.read_csv('../outputs/y_train.csv').squeeze()
y_test  = pd.read_csv('../outputs/y_test.csv').squeeze()

print(f'X_train : {X_train.shape}  | y_train : {dict(y_train.value_counts())}')
print(f'X_test  : {X_test.shape}   | y_test  : {dict(y_test.value_counts())}')

## 2. Définition des 7 modèles

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'),

    'K-Nearest Neighbors': KNeighborsClassifier(
        n_neighbors=7, metric='euclidean'),

    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1),

    'SVM (RBF)': SVC(
        kernel='rbf', probability=True, random_state=42, class_weight='balanced'),

    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42),

    'AdaBoost': AdaBoostClassifier(
        n_estimators=100, learning_rate=0.5, random_state=42),

    'MLP Neural Network': MLPClassifier(
        hidden_layer_sizes=(64, 32), activation='relu',
        max_iter=500, random_state=42, early_stopping=True),
}
print(f'✅ {len(models)} modèles définis')

## 3. Entraînement et évaluation de tous les modèles

In [ ]:
results = {}

for name, model in models.items():
    print(f'⏳ {name}...')
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    cv_auc = cross_val_score(model, X_train, y_train,
                              cv=cv, scoring='roc_auc', n_jobs=-1)
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    prec, rec, _ = precision_recall_curve(y_test, y_prob)

    results[name] = {
        'model'     : model,
        'y_pred'    : y_pred,
        'y_prob'    : y_prob,
        'accuracy'  : accuracy_score(y_test, y_pred),
        'auc'       : roc_auc_score(y_test, y_prob),
        'f1'        : f1_score(y_test, y_pred, zero_division=0),
        'recall'    : recall_score(y_test, y_pred, zero_division=0),
        'precision' : precision_score(y_test, y_pred, zero_division=0),
        'cv_auc'    : cv_auc.mean(),
        'cv_std'    : cv_auc.std(),
        'cm'        : confusion_matrix(y_test, y_pred),
        'fpr'       : fpr, 'tpr': tpr,
        'prec_curve': prec, 'rec_curve': rec,
        'ap'        : average_precision_score(y_test, y_prob),
    }
    print(f'  ✅ AUC={results[name]["auc"]:.4f} | F1={results[name]["f1"]:.4f} | Recall={results[name]["recall"]:.4f}')

print('\n✅ Tous les modèles entraînés !')

## 4. Tableau comparatif

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Modèle'    : name,
        'AUC-ROC'   : round(r['auc'], 4),
        'CV AUC'    : f"{r['cv_auc']:.4f} ± {r['cv_std']:.4f}",
        'F1'        : round(r['f1'], 4),
        'Recall'    : round(r['recall'], 4),
        'Precision' : round(r['precision'], 4),
        'Accuracy'  : round(r['accuracy'], 4),
    })
df_res = pd.DataFrame(rows).sort_values('AUC-ROC', ascending=False).reset_index(drop=True)

print('=' * 72)
print('  CLASSEMENT DES MODÈLES (par AUC-ROC décroissant)')
print('=' * 72)
print(df_res.to_string(index=False))
print('=' * 72)

best_name = df_res.iloc[0]['Modèle']
print(f'\n🏆 Meilleur modèle : {best_name}')

## 5. Courbes ROC — tous les modèles

In [ ]:
colors = ['#3498db','#e74c3c','#2ecc71','#9b59b6','#f39c12','#1abc9c','#e67e22']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ROC
for (name, r), c in zip(results.items(), colors):
    lw = 2.5 if name == best_name else 1.2
    al = 1.0 if name == best_name else 0.6
    axes[0].plot(r['fpr'], r['tpr'], color=c, lw=lw, alpha=al,
                 label=f"{name} ({r['auc']:.3f})")
axes[0].plot([0,1],[0,1],'k--',lw=0.8, label='Aléatoire')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Comparaison', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')
axes[0].grid(alpha=0.3)

# Precision-Recall
for (name, r), c in zip(results.items(), colors):
    lw = 2.5 if name == best_name else 1.2
    axes[1].plot(r['rec_curve'], r['prec_curve'], color=c, lw=lw,
                 label=f"{name} (AP={r['ap']:.3f})")
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/roc_pr_curves.png', dpi=150)
plt.show()

## 6. Matrices de confusion

In [ ]:
n = len(results)
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

for i, (name, r) in enumerate(results.items()):
    sns.heatmap(r['cm'], annot=True, fmt='d', ax=axes[i],
                cmap='Blues', linewidths=1, linecolor='white',
                xticklabels=['No Cancer','Cancer'],
                yticklabels=['No Cancer','Cancer'],
                annot_kws={'size':13,'weight':'bold'})
    auc_str = f"AUC={r['auc']:.3f}"
    axes[i].set_title(f"{name}\n{auc_str}", fontsize=9, fontweight='bold')
    axes[i].set_xlabel('Prédit', fontsize=8)
    axes[i].set_ylabel('Réel', fontsize=8)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Matrices de Confusion — Tous les modèles', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Comparaison des métriques — Barplot

In [ ]:
metrics_list = ['auc','f1','recall','precision','accuracy']
labels       = ['AUC-ROC','F1','Recall','Precision','Accuracy']

x     = np.arange(len(metrics_list))
width = 0.11
colors= ['#3498db','#e74c3c','#2ecc71','#9b59b6','#f39c12','#1abc9c','#e67e22']

fig, ax = plt.subplots(figsize=(15, 6))
for i, (name, r) in enumerate(results.items()):
    vals = [r[m] for m in metrics_list]
    bars = ax.bar(x + i*width, vals, width, label=name,
                  color=colors[i], alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.2f}', ha='center', va='bottom', fontsize=6, fontweight='bold')

ax.set_xticks(x + width * (len(results)-1)/2)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Comparaison complète des métriques — 7 modèles', fontsize=13, fontweight='bold')
ax.legend(fontsize=8, loc='upper right', ncol=2)
ax.grid(axis='y', alpha=0.3)
ax.axhline(0.9, color='gray', linestyle='--', lw=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig('../outputs/metrics_comparison.png', dpi=150)
plt.show()

## 8. Rapport détaillé du meilleur modèle

In [ ]:
best = results[best_name]
tn, fp, fn, tp = best['cm'].ravel()

print(f'{'='*55}')
print(f'  RAPPORT DÉTAILLÉ — {best_name}')
print(f'{'='*55}')
print(classification_report(
    y_test, best['y_pred'],
    target_names=['No Cancer (0)','Cancer (1)'], digits=4))
print(f'  Vrais Positifs  (TP) : {tp}  — cancers détectés ✅')
print(f'  Vrais Négatifs  (TN) : {tn}  — non-cancers corrects ✅')
print(f'  Faux Positifs   (FP) : {fp}   — fausses alarmes ⚠️')
print(f'  Faux Négatifs   (FN) : {fn}   — cancers MANQUÉS 🚨')
print(f'  Sensibilité          : {tp/(tp+fn):.4f}')
print(f'  Spécificité          : {tn/(tn+fp):.4f}')

## 9. Hyperparameter Tuning du meilleur modèle

In [ ]:
param_grids = {
    'Random Forest': {
        'n_estimators'     : [100, 200, 300, 400],
        'max_depth'        : [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf' : [1, 2, 4],
        'max_features'     : ['sqrt', 'log2'],
    },
    'Gradient Boosting': {
        'n_estimators'  : [100, 200, 300],
        'learning_rate' : [0.01, 0.05, 0.1, 0.2],
        'max_depth'     : [3, 4, 5, 6],
        'subsample'     : [0.7, 0.8, 0.9, 1.0],
        'min_samples_split': [2, 5, 10],
    },
    'SVM (RBF)': {
        'C'    : [0.1, 1, 10, 100],
        'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
    },
    'Logistic Regression': {
        'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver' : ['liblinear', 'saga'],
    },
    'MLP Neural Network': {
        'hidden_layer_sizes': [(32,), (64,32), (128,64), (64,32,16)],
        'alpha'             : [0.0001, 0.001, 0.01],
        'learning_rate_init': [0.001, 0.01],
    },
    'AdaBoost': {
        'n_estimators' : [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5, 1.0],
    },
    'K-Nearest Neighbors': {
        'n_neighbors': [3, 5, 7, 9, 11, 15],
        'metric'     : ['euclidean', 'manhattan', 'minkowski'],
        'weights'    : ['uniform', 'distance'],
    },
}

if best_name in param_grids:
    print(f'⏳ RandomizedSearchCV pour {best_name} (n_iter=25)...')
    search = RandomizedSearchCV(
        results[best_name]['model'].__class__(
            **{k:v for k,v in results[best_name]['model'].get_params().items()
               if k in ['random_state','probability','class_weight','n_jobs']
               and v is not None}
        ),
        param_grids[best_name],
        n_iter=25,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring='roc_auc',
        n_jobs=-1,
        random_state=42,
        verbose=0
    )
    search.fit(X_train, y_train)
    best_tuned = search.best_estimator_

    y_pred_t = best_tuned.predict(X_test)
    y_prob_t = best_tuned.predict_proba(X_test)[:, 1]

    print(f'\n✅ Meilleurs paramètres : {search.best_params_}')
    print(f'\nComparaison avant/après tuning :')
    print(f'  AUC  avant : {results[best_name]["auc"]:.4f}  →  après : {roc_auc_score(y_test, y_prob_t):.4f}')
    print(f'  F1   avant : {results[best_name]["f1"]:.4f}  →  après : {f1_score(y_test, y_pred_t, zero_division=0):.4f}')
    print(f'  Rec  avant : {results[best_name]["recall"]:.4f}  →  après : {recall_score(y_test, y_pred_t, zero_division=0):.4f}')
else:
    best_tuned = results[best_name]['model']
    print(f'Pas de grille pour {best_name}, on garde le modèle de base.')

## 10. Sauvegarde du meilleur modèle

In [ ]:
joblib.dump(best_tuned, '../outputs/best_model_tuned.pkl')
joblib.dump(best_name,  '../outputs/best_model_name.pkl')

# Aussi sauvegarder tous les modèles
all_models = {name: r['model'] for name, r in results.items()}
joblib.dump(all_models, '../outputs/all_models.pkl')

print(f'✅ Meilleur modèle tuné sauvegardé : {best_name}')
print('✅ Tous les modèles sauvegardés dans outputs/all_models.pkl')
print('\n✅ Experiments terminés → lancer 04_Final_Notebook.ipynb')